## Step 3 - Reduce keyframes to webp

Downscales the full-res jpg keyframes from Step 2 and re-encodes them as small
`.webp` files for serving. Layout is nested by batch and the basename is
unchanged, so it lines up with `frames_meta.json`:

```
Keyframes/<batch>/<video_name>/<idx:06d>.jpg  ->  Keyframes/<batch>/<video_name>/<idx:06d>.webp
e.g. Keyframes/L21/L21_V001/000080.jpg        ->  Keyframes/L21/L21_V001/000080.webp
```

Must use the same extension recorded as `FINAL_EXT` in Step 2 (`webp`).

In [5]:
import os
import math
import glob
from PIL import Image
from tqdm import tqdm

In [ ]:
SRC_DIR      = './Keyframes'   # output of Step 2 (flat jpg)
SAVE_DIR     = '/mnt/data2tb/code/Github/AIC/Ho-Chi-Minh-AI-Challenge/backend/media/frames/Keyframes'
REDUCE_RATIO = 0.5
WEBP_QUALITY = 70

os.makedirs(SAVE_DIR, exist_ok=True)

In [7]:
# Auto-discover every video folder (nested: <batch>/<video_name>/*.jpg)
video_dirs = sorted(
    (batch, vid)
    for batch in os.listdir(SRC_DIR)
    if os.path.isdir(os.path.join(SRC_DIR, batch))
    for vid in os.listdir(os.path.join(SRC_DIR, batch))
    if os.path.isdir(os.path.join(SRC_DIR, batch, vid))
)
print('video folders:', len(video_dirs))

video folders: 60


In [9]:
for batch, video_name in tqdm(video_dirs):
    in_dir  = os.path.join(SRC_DIR, batch, video_name)
    out_dir = os.path.join(SAVE_DIR, batch, video_name)   # nested: Keyframes/L21/L21_V001
    os.makedirs(out_dir, exist_ok=True)

    for image_path in sorted(glob.glob(f'{in_dir}/*.jpg')):
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        out_path = os.path.join(out_dir, f'{image_id}.webp')
        if os.path.exists(out_path):
            continue
        img = Image.open(image_path).convert('RGB')
        x, y = img.size
        img = img.resize((math.floor(x * REDUCE_RATIO), math.floor(y * REDUCE_RATIO)))
        img.save(out_path, format='webp', optimize=True, quality=WEBP_QUALITY)
print('done')

100%|██████████| 60/60 [29:42<00:00, 29.71s/it]

done
